In [ ]:
# Make local packages visible
push!(LOAD_PATH, joinpath(pwd(), "src/LorentzianSimplexSolver", "src"))

using LorentzianSimplexSolver

In [2]:
include("run_geometry.jl");
include("run_action.jl")
include("run_dlogEh_dg.jl")
include("run_djdl.jl")
include("TransverseBasis.jl")
include("LinearizedEOMs.jl")
include("QuadraticRegge.jl")

Main.QuadraticRegge

In [3]:
using JLD2, PythonCall
using Symbolics
@variables γ;

using .RunGeometry
using .RunAction
using .RunDlogEhDG
using .DJDLUtils
using .TransverseBasis
using .QuadraticRegge

#### Geometry setup

In [4]:
simplices = [[1,2,3,4,6],[1,2,3,5,6],[1,2,4,5,6],[1,3,4,5,6],[2,3,4,5,6]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

coords_lines = [
    "0, 0, 0, 0",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "-0.068,-0.27,-0.5,-1.3",
]

const ScalarT = Float64
tol = 1e-10;
gamma_vals = one(ScalarT);

vertex_coords = Dict{Int, Vector{ScalarT}}()  

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [5]:
geom_base = run_geometry_pipeline(simplices, coords_lines, ScalarT, tol);

#### Action and variables calculation

In [6]:
sd_base, S_base_fn, args_base, labels_base = RunAction.run_action(geom_base, γ);

#### Hessian matrix computation or read Hessian from files

In [7]:
filepath = joinpath(pwd(), "Hessian_data/H_base_eval_15move.jld2")

if isfile(filepath)
    @load filepath H_base_eval
else
    include("Hessian_data/run_hessian.jl")
    using .RunHessian
    run_hessian(geom_base, gamma_vals, filepath)
end;

#### Computation of $\frac{\partial \log E_h}{\partial g_\alpha}$

In [8]:
g_vars = geom_base.varias[:g_var]
nh = length(geom_base.connectivity[1]["OrderBulkFaces"])

dlogEh_dg_func = RunDlogEhDG.run_dlogEh_dg(geom_base, sd_base);
args_vals = substitute.(args_base, Ref(γ => gamma_vals));
# dlogEh_dg_vals is a nh x ng matrix
dlogEh_dg_vals = transpose(hcat([[dlogEh_dg_func[h][v](args_vals...) for v in g_vars] for h in 1:nh]...));

#### Computation of $\frac{\partial j_h}{\partial \ell_s}$

In [9]:
# djdl_matrix is a nh x nl matrix
djdl_matrix, bulk_edges, j_h_vertices = DJDLUtils.build_djdl_matrix(geom_base, vertex_coords, ScalarT, gamma_vals);
nl = length(bulk_edges)
nt = nh - nl;

#### Computation of $\hat{e}^i_h$

In [10]:
# eListHT is a nh x nt matrix
eListHT = TransverseBasis.compute_transverse_basis(djdl_matrix, tol);

#### computation of Linearized solution to EOMs

In [11]:
z_vars = geom_base.varias[:z_var]

j_vars = [
    LorentzianSimplexSolver.DefineSymbols._sympy().symbols("j_$(a)_$(b)_$(c)", real=true)
    for v in geom_base.connectivity[1]["OrderBulkFaces"]
    for (a,b,c) in (Tuple(v[1]),)
]

vars = vcat(g_vars, z_vars, j_vars)
labels = sd_base.labels_vars
idx_X = [findfirst(l -> pyis(l, X), labels) for X in vars]

ng = length(g_vars)
nz = length(z_vars)
nx = ng + nz

HessianOld = H_base_eval[idx_X, idx_X]
@variables dl[1:nl]
dl_vec = collect(dl) 
dYsoln, HYY = LinearizedEOMs.solve_linearized_eoms(HessianOld, eListHT, djdl_matrix, nx, ScalarT, dl_vec);

#### Computation of $W_{h_1 h_2}$ matrix, $S_{ij}$ matrix and $\kappa_{h_1 h_2}$ matrix

In [12]:
invHessianXX = inv(HessianOld[1:nx, 1:nx])[1:ng, 1:ng]
Wmatrix = -dlogEh_dg_vals * invHessianXX * transpose(dlogEh_dg_vals);

Smatrix = eListHT' * Wmatrix * eListHT;
Smatrix_num = ComplexF64.(Symbolics.value.(Smatrix));

kappa_matrix = eListHT * inv(Smatrix_num) * eListHT';
quadratic_correct = -1/2 .* djdl_matrix' * Wmatrix * kappa_matrix * transpose(Wmatrix) * djdl_matrix;

In [13]:
W_all_matrix = -dlogEh_dg_vals * inv(HYY)[1:ng, 1:ng] * transpose(dlogEh_dg_vals);
Spinfoam_Quadratic1 = 1/2 * transpose(djdl_matrix) * W_all_matrix * djdl_matrix

5×5 Matrix{Any}:
 -10.3619 - 0.28771im    …    -2.8283 - 0.0785306im
 -1.16749 - 0.0324164im     -0.318666 - 0.00884809im
 -7.19477 - 0.19977im        -1.96382 - 0.0545275im
 -3.54319 - 0.0983802im     -0.967117 - 0.026853im
  -2.8283 - 0.0785306im     -0.771987 - 0.021435im

#### Computation of Quadratic term of Regge action $iS^{(2)}_{\mathrm{Regge}}$

In [14]:
dthetadl = QuadraticRegge.compute_dθDl(simplices, j_h_vertices, bulk_edges, vertex_coords, geom_base.connectivity[1]["Tets"], LorentzianSimplexSolver.Dihedral.minkowski_norm2);
dadl = gamma_vals .* djdl_matrix;
ReggeQuadratic = im*1/2 * dadl' * dthetadl;

#### Computation of Quadratic term correction $\frac{\gamma^2}{2} \kappa_{h_1 h_2}\delta\epsilon_{h_1}\,\delta\epsilon_{h_2}$ to $iS^{(2)}_{\mathrm{Regge}}$

In [15]:
correction = gamma_vals^2/2 * dthetadl' * kappa_matrix * dthetadl;

#### Computation of Spinfoam Quadratic term $\mathcal I_{\mathrm{eff}}(\ell)-\mathring S$

In [16]:
Spinfoam_Quadratic = ReggeQuadratic + correction

5×5 Matrix{ComplexF64}:
 -10.3619-0.28771im     -1.16749-0.0324164im   …    -2.8283-0.0785306im
 -1.16749-0.0324164im  -0.131541-0.00365238im     -0.318666-0.00884809im
 -7.19477-0.19977im    -0.810639-0.0225082im       -1.96382-0.0545275im
 -3.54319-0.0983802im  -0.399213-0.0110846im      -0.967117-0.026853im
  -2.8283-0.0785306im  -0.318666-0.00884809im     -0.771987-0.021435im